## Read & write en Delta Lake
1. Escribri datos en Delta Lake (Managed Table)
2. Escribir datos en Delta Lake (External Table)
3. Leer datos desde Delta Lake (Table)
4. Leer datos desde Delta Lake (Files)

In [0]:
%sql
create schema if not exists movie_demo
managed location "abfss://demo@moviehistorytesteo.dfs.core.windows.net/"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

In [0]:
movie_schema = StructType( fields= [
    StructField("movieId", IntegerType(), False),
    StructField("title",StringType(), True),
    StructField("budget",IntegerType(), True),
    StructField("homePage",StringType(), True),
    StructField("overview",StringType(), True),
    StructField("popularity",DoubleType(), True),
    StructField("yearReleaseDate",IntegerType(), True),
    StructField("releaseDate",DateType(), True),
    StructField("revenue",IntegerType(), True),
    StructField("durationTime",IntegerType(), True),
    StructField("movieStatus",StringType(), True),
    StructField("tagline",StringType(), True),
    StructField("voteAverage",DoubleType(), True),
    StructField("voteCount",IntegerType(), True)
])

In [0]:
movie_df = spark.read.options(header = True).schema(movie_schema).csv("abfss://bronze@moviehistorytesteo.dfs.core.windows.net/2024-12-30/movie.csv")

In [0]:
display(movie_df)

In [0]:
movie_df.write.format("delta").mode("overwrite").saveAsTable("movie_demo.movies_managed")

In [0]:
%sql
select * from movie_demo.movies_managed

In [0]:
movie_df.write.format("delta").mode("overwrite").save("abfss://demo@moviehistorytesteo.dfs.core.windows.net/movies_external")

In [0]:
%sql
create table movie_demo.movies_external
using delta
location "abfss://demo@moviehistorytesteo.dfs.core.windows.net/movies_external"

In [0]:
%sql
select * from movie_demo.movies_external

In [0]:
movie_df.write.format("delta").mode("overwrite").partitionBy("yearReleaseDate").saveAsTable("movie_demo.movies_partitioned")

In [0]:
%sql
select * from movie_demo.movies_partitioned; 

In [0]:
%sql 
desc extended movie_demo.movies_partitioned

In [0]:
%sql
show partitions movie_demo.movies_partitioned;

## Updqate & Delete en Delta Lake
1. Update desde Delta Lake
2. Deleate desde Delta Lake

In [0]:
%sql
update movie_demo.movies_managed
set durationTime = 60
where yearReleaseDate = 2012

In [0]:
%sql 
select * from movie_demo.movies_managed
where yearReleaseDate = 2014


In [0]:
from delta.tables import *

deltaTable = DeltaTable.forName(spark, 'movie_demo.movies_managed')

# Declare the predicate by using a SQL-formatted string.
deltaTable.update(
  condition = "yearReleaseDate = '2013'",
  set = { "durationTime": "100" }
)

In [0]:
%sql 
select * from movie_demo.movies_managed
where yearReleaseDate = 2013

## Delete Registers

In [0]:
%sql 
Delete from movie_demo.movies_managed
where yearReleaseDate = 2014

In [0]:
%sql 
select * from movie_demo.movies_managed
where yearReleaseDate = 2014

In [0]:
from delta.tables import *

deltaTable = DeltaTable.forName(spark, 'movie_demo.movies_managed')

# Declare the predicate by using a SQL-formatted string.
deltaTable.delete("yearReleaseDate = 2015")

In [0]:
%sql 
select * from movie_demo.movies_managed
where yearReleaseDate = 2015

## Merge / Upsert en Delta Lake

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

In [0]:
movie_schema = StructType( fields= [
    StructField("movieId", IntegerType(), False),
    StructField("title",StringType(), True),
    StructField("budget",IntegerType(), True),
    StructField("homePage",StringType(), True),
    StructField("overview",StringType(), True),
    StructField("popularity",DoubleType(), True),
    StructField("yearReleaseDate",IntegerType(), True),
    StructField("releaseDate",DateType(), True),
    StructField("revenue",IntegerType(), True),
    StructField("durationTime",IntegerType(), True),
    StructField("movieStatus",StringType(), True),
    StructField("tagline",StringType(), True),
    StructField("voteAverage",DoubleType(), True),
    StructField("voteCount",IntegerType(), True)
])

In [0]:
movie_day1_df = spark.read.options(header = True).schema(movie_schema).csv("abfss://bronze@moviehistorytesteo.dfs.core.windows.net/2024-12-30/movie.csv")\
    .filter("yearReleaseDate < 2000")\
    .select("movieId","title","yearReleaseDate","releaseDate","durationTime")


In [0]:
display(movie_day1_df)


In [0]:
movie_day1_df.createOrReplaceTempView("v_movie_day1_df")

In [0]:
from pyspark.sql.functions import upper

movie_day2_df = spark.read\
    .options(header = True)\
    .schema(movie_schema)\
    .csv("abfss://bronze@moviehistorytesteo.dfs.core.windows.net/2024-12-30/movie.csv")\
    .filter("yearReleaseDate between 1998 and 2005")\
    .select("movieId",upper("title").alias("title"),"yearReleaseDate","releaseDate","durationTime")

In [0]:
display(movie_day2_df)

In [0]:
movie_day2_df.createOrReplaceTempView("v_movie_day2_df")

In [0]:
from pyspark.sql.functions import upper

movie_day3_df = spark.read\
    .options(header = True)\
    .schema(movie_schema)\
    .csv("abfss://bronze@moviehistorytesteo.dfs.core.windows.net/2024-12-30/movie.csv")\
    .filter("yearReleaseDate between 1983 and 1998 or yearReleaseDate between 2006 and 2010")\
    .select("movieId",upper("title").alias("title"),"yearReleaseDate","releaseDate","durationTime")

In [0]:
display(movie_day3_df)

In [0]:
%sql 
create table if not exists movie_demo.movie_merge(
  movieId int,
  title string,
  yearReleaseDate int,
  releaseDate date,
  durationTime int,
  createdDate date,
  updatedDate date
)

In [0]:
#data procesada a partir del dia 1 que se cargo por un temview

In [0]:
%sql
MERGE INTO movie_demo.movie_merge tgt
USING v_movie_day1_df src
ON tgt.movieId = src.movieId
WHEN MATCHED THEN
  UPDATE SET 
    tgt.title = src.title,
    tgt.yearReleaseDate = src.yearReleaseDate,
    tgt.releaseDate = src.releaseDate,
    tgt.durationTime = src.durationTime, 
    tgt.updatedDate = current_timestamp
WHEN NOT MATCHED THEN
  INSERT (tgt.movieId,tgt.title,tgt.yearReleaseDate,tgt.releaseDate,tgt.durationTime,tgt.createdDate)
  VALUES (src.movieId,src.title,src.yearReleaseDate,src.releaseDate,src.durationTime,current_timestamp)

In [0]:
#Dia 2 a partir de un temview

In [0]:
%sql
MERGE INTO movie_demo.movie_merge tgt
USING v_movie_day2_df src
ON tgt.movieId = src.movieId
WHEN MATCHED THEN
  UPDATE SET 
    tgt.title = src.title,
    tgt.yearReleaseDate = src.yearReleaseDate,
    tgt.releaseDate = src.releaseDate,
    tgt.durationTime = src.durationTime, 
    tgt.updatedDate = current_timestamp
WHEN NOT MATCHED THEN
  INSERT (tgt.movieId,tgt.title,tgt.yearReleaseDate,tgt.releaseDate,tgt.durationTime,tgt.createdDate)
  VALUES (src.movieId,src.title,src.yearReleaseDate,src.releaseDate,src.durationTime,current_timestamp)

In [0]:
# cragaremos el merge a partir de codigo Python, no en TemView como las demas

In [0]:
from delta.tables import *

deltaTablePeople = DeltaTable.forName(spark, 'movie_demo.movie_merge')

deltaTablePeople.alias('tgt') \
  .merge(
    movie_day3_df.alias('src'),
    'tgt.movieId = src.movieId'
  ) \
  .whenMatchedUpdate(set =
    {
      "tgt.title": "src.title",
      "tgt.yearReleaseDate": "src.yearReleaseDate",
      "tgt.releaseDate": "src.releaseDate",
      "tgt.durationTime": "src.durationTime",
      "tgt.updatedDate" : "current_timestamp()"
    }
  ) \
  .whenNotMatchedInsert(values =
    {
      "tgt.movieId": "src.movieId",
      "tgt.title": "src.title",
      "tgt.yearReleaseDate": "src.yearReleaseDate",
      "tgt.releaseDate": "src.releaseDate",
      "tgt.durationTime": "src.durationTime",
      "tgt.createdDate" : "current_timestamp()"
    }
  ) \
  .execute()

In [0]:
%sql
select * from movie_demo.movie_merge;

## History,Time Travel, Vacuum
1. Historial y control de versiones
2. Viajes en el tiempo
3. Vacio


In [0]:
%sql
select * from movie_demo.movie_merge version as of 2

In [0]:
## visualizamos es historial de versiones de la tabla "merge"

In [0]:
%sql
desc history movie_demo.movie_merge

In [0]:
df=spark.read.format("delta").option("timestamAsOf","2026-03-30T23:11:34.000+00:00").table("movie_demo.movie_merge")

In [0]:
display(df)

In [0]:
%sql
vacuum movie_demo.movie_merge
  

In [0]:
%sql
desc history movie_demo.movie_merge

In [0]:
### frozaremos la eliminación de la historia de las veciones

In [0]:
%sql
SET spark.databricks.delta.retentionDurationCheck.enabled = false;
VACUUM movie_demo.movie_merge RETAIN 0 HOURS;

In [0]:
%sql
desc history movie_demo.movie_merge

In [0]:
%sql
select * from movie_demo.movie_merge version as of 2

In [0]:
%sql
DELETE FROM movie_demo.movie_merge 
where yearReleaseDate = 2004

In [0]:
%sql
SELECT * FROM movie_demo.movie_merge;


In [0]:
%sql 
DESC HISTORY movie_demo.movie_merge;

In [0]:
%sql
MERGE INTO movie_demo.movie_merge tgt
USING movie_demo.movie_merge VERSION AS OF 10 src
ON tgt.movieId = src.movieId
WHEN NOT MATCHED THEN
  INSERT *

In [0]:
%sql
SELECT * FROM movie_demo.movie_merge;

In [0]:
%sql
DESC HISTORY movie_demo.movie_merge;

## Transaccion Log en Delta Lake
#### Con esto podremos saber con presicion donde se almacenan los datos, ya que estos datos de recuperación como el History no se almacenan en el Unity Catalog.

crearemos una tabla en movie_demo para poder hacer la demostración llamada "movie_log"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS movie_demo.movie_log(
  movieId int,
  title string,
  yearReleaseDate int,
  releaseDate date,
  durationTime int,
  createdDate date,
  updatedDate date
)
USING DELTA

In [0]:
%sql
DESC HISTORY movie_demo.movie_log;

In [0]:
%sql
DESC EXTENDED movie_demo.movie_log;

In [0]:
%sql
INSERT INTO movie_demo.movie_log
SELECT * FROM   movie_demo.movie_merge
where movieId = 125537

In [0]:
%sql
DESC HISTORY movie_demo.movie_log;

In [0]:
%sql
INSERT INTO movie_demo.movie_log
SELECT * FROM   movie_demo.movie_merge
where movieId = 133575

In [0]:
%sql
DESC HISTORY movie_demo.movie_log;

## Convertir formato "Parquet" a "delta"


In [0]:
%sql
CREATE TABLE IF NOT EXISTS movie_demo.movie_convert_to_delta(
  movieId int,
  title string,
  yearReleaseDate int,
  releaseDate date,
  durationTime int,
  createdDate date,
  updatedDate date
)
USING PARQUET
LOCATION 'abfss://demo@moviehistorytesteo.dfs.core.windows.net/movies_convert_to_delta'

In [0]:
%sql
INSERT INTO movie_demo.movie_convert_to_delta
SELECT * FROM  movie_demo.movie_merge;

In [0]:
%sql
CONVERT TO DELTA movie_demo.movie_convert_to_delta;

In [0]:
df= spark.table("movie_demo.movie_convert_to_delta")

In [0]:
display(df)

In [0]:
df.write.format("parquet").save("abfss://demo@moviehistorytesteo.dfs.core.windows.net/movies_convert_to_delta_new")

In [0]:
%sql 
CONVERT TO DELTA parquet.`abfss://demo@moviehistorytesteo.dfs.core.windows.net/movies_convert_to_delta_new`